# Сравнение запусков DLightSB и DLightSB-MM

Ноутбук принимает два или более ZIP-архива сохранённых запусков

Можно одновременно сравнивать несколько запусков одного и того же метода

Подписи берутся из `experiment_tag` внутри архива или из имени ZIP-файла

## Импорты и оформление


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (9, 5),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "lines.linewidth": 2,
        "lines.markersize": 6,
    }
)


## Выбор архивов

Выберите два или более ZIP-архива

Архивы должны относиться к одной размерности

In [ ]:
import tkinter as tk
from tkinter import filedialog


dialog = tk.Tk()
dialog.withdraw()
dialog.attributes("-topmost", True)

try:
    selected_files = filedialog.askopenfilenames(
        title="Выберите два или более архива catsbench",
        filetypes=[("ZIP archives", "*.zip")],
    )
finally:
    dialog.destroy()

ARCHIVE_PATHS = [
    Path(path).resolve()
    for path in selected_files
]

if len(ARCHIVE_PATHS) < 2:
    raise ValueError(
        f"Нужно выбрать не менее двух ZIP-архивов выбрано {len(ARCHIVE_PATHS)}"
    )

print("Выбраны архивы:")
for path in ARCHIVE_PATHS:
    print("-", path)

## Распаковка и чтение результатов


In [ ]:
ANALYSIS_ROOT = Path.cwd().resolve() / ".catsbench_model_analysis"

if ANALYSIS_ROOT.is_dir():
    shutil.rmtree(ANALYSIS_ROOT)

ANALYSIS_ROOT.mkdir(parents=True)


def extract_archive(archive_path: Path, extract_dir: Path) -> None:
    extract_dir.mkdir(parents=True)
    root = extract_dir.resolve()

    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (extract_dir / member.filename).resolve()

            if not target.is_relative_to(root):
                raise ValueError(
                    f"Некорректный путь внутри архива: {member.filename}"
                )

        archive.extractall(extract_dir)


EXTRACTED_ARCHIVES = []

for index, archive_path in enumerate(ARCHIVE_PATHS):
    extract_dir = ANALYSIS_ROOT / f"{index}_{archive_path.stem}"
    extract_archive(archive_path, extract_dir)
    EXTRACTED_ARCHIVES.append((archive_path, extract_dir))
    print(f"{archive_path.name} распакован в {extract_dir}")


In [ ]:
MODEL_LABELS = {
    "baseline": "Gradient descent",
    "mm": "MM",
    "mm_regularized": "MM regularized",
}

MODEL_COLORS = {}


def find_one(run_dir: Path, pattern: str) -> Path:
    matches = sorted(run_dir.glob(pattern))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Для {pattern} ожидался один файл найдено {len(matches)}"
        )

    return matches[0]


def find_logger_file(run_dir: Path, logger_name: str, relative_path: str) -> Path:
    preferred = run_dir / logger_name / relative_path

    if preferred.is_file():
        return preferred

    return find_one(run_dir, f"*/{relative_path}")


def read_run(run_dir: Path) -> dict:
    train_metrics_path = find_logger_file(run_dir, "csv", "train/metrics.csv")
    test_metrics_path = find_logger_file(run_dir, "csv", "test/metrics.csv")
    hparams_path = find_logger_file(run_dir, "csv", "train/hparams.yaml")

    with hparams_path.open(encoding="utf-8") as file:
        config = yaml.safe_load(file)

    return {
        "run_dir": run_dir,
        "train": pd.read_csv(train_metrics_path),
        "test": pd.read_csv(test_metrics_path),
        "config": config,
    }


def archived_models(extract_dir: Path, manifest: dict) -> list[tuple[str, Path]]:
    if manifest.get("archive_format") == "single_run_v1":
        kind = manifest["kind"]
        return [(kind, extract_dir / kind)]

    if "experiments" in manifest:
        return [
            (kind, extract_dir / kind)
            for kind in MODEL_LABELS
            if kind in manifest["experiments"]
        ]

    raise ValueError("Неизвестный формат архива")


def unique_run_label(kind: str, archive_label: str, existing: dict) -> str:
    base = f"{MODEL_LABELS[kind]} · {archive_label}"
    label = base
    suffix = 2

    while label in existing:
        label = f"{base} · {suffix}"
        suffix += 1

    return label


runs = {}
metadata_rows = {}

for archive_path, extract_dir in EXTRACTED_ARCHIVES:
    with (extract_dir / "manifest.json").open(encoding="utf-8") as file:
        manifest = json.load(file)

    archive_label = str(
        manifest.get("experiment_tag") or archive_path.stem
    ).strip()

    for kind, run_dir in archived_models(extract_dir, manifest):
        if kind not in MODEL_LABELS:
            raise ValueError(f"Неизвестный тип модели: {kind}")

        label = unique_run_label(kind, archive_label, runs)
        run = read_run(run_dir)
        run["kind"] = kind
        runs[label] = run
        metadata_rows[label] = {
            "model": MODEL_LABELS[kind],
            "experiment_tag": manifest.get("experiment_tag"),
            "dimension": int(manifest["dimension"]),
            "seed": int(manifest["seed"]),
            "commit": manifest["commit"],
            "archive": archive_path.name,
        }

if not runs:
    raise RuntimeError("В архивах не найдено ни одного запуска")

dimensions = {row["dimension"] for row in metadata_rows.values()}

if len(dimensions) != 1:
    raise ValueError(f"Размерности запусков не совпадают: {sorted(dimensions)}")

DIMENSION = dimensions.pop()
color_map = plt.get_cmap("tab20")
MODEL_COLORS.update({
    label: color_map(index % color_map.N)
    for index, label in enumerate(runs)
})
metadata = pd.DataFrame.from_dict(metadata_rows, orient="index")
display(metadata.style.set_caption("Загруженные запуски"))


## Финальные метрики

Для score и effective k лучшим считается большее значение

Для loss и log c лучшим считается меньшее значение

Для log v лучшим считается большее значение поскольку `loss = log_c - log_v`


In [ ]:
SUMMARY_COLUMNS = [
    "cond_shape_score_forward",
    "cond_trend_score_forward",
    "loss",
    "log_c",
    "log_v",
    "posterior_eff_k",
    "prior_eff_k",
    "shape_score_forward",
    "trend_score_forward",
]

MINIMIZE_COLUMNS = {"loss", "log_c"}
BEST_COLOR = "#6f42c1"


def final_test_values(frame: pd.DataFrame) -> dict:
    values = {}

    for column in frame.columns:
        if not column.startswith("test/"):
            continue

        series = frame[column].dropna()

        if len(series):
            values[column.removeprefix("test/")] = float(series.iloc[-1])

    if "log_c" not in values or "log_v" not in values:
        raise KeyError("В test CSV отсутствуют log_c или log_v")

    values["loss"] = values["log_c"] - values["log_v"]
    return values


def highlight_best(frame: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)

    if len(frame.index) < 2:
        return styles

    for column in frame.columns:
        valid = frame[column].dropna()

        if valid.empty:
            continue

        best = valid.min() if column in MINIMIZE_COLUMNS else valid.max()
        winners = np.isclose(frame[column], best, rtol=1e-9, atol=1e-12)
        styles.loc[winners, column] = (
            f"background-color: {BEST_COLOR}; color: white; font-weight: bold"
        )

    return styles


In [ ]:
summary = pd.DataFrame(
    {
        label: final_test_values(run["test"])
        for label, run in runs.items()
    }
).T

missing_columns = [
    column
    for column in SUMMARY_COLUMNS
    if column not in summary.columns
]

if missing_columns:
    raise KeyError(
        "В итоговых логах отсутствуют метрики: "
        + ", ".join(missing_columns)
    )

summary = summary[SUMMARY_COLUMNS]

display(
    summary.style
    .format(precision=5, na_rep="—")
    .apply(highlight_best, axis=None)
    .set_caption("Финальные test-метрики")
)


## Loss без штрафа по числу примеров

Основной Loss использует `train/data_loss` при наличии этой колонки

Для обычного MM и baseline используется `train/loss` или разность `train/log_c - train/log_v`

Regularized loss — отдельная сумма Loss и штрафа из `train/loss` регуляризованного MM


In [ ]:
def last_non_null(series: pd.Series):
    values = series.dropna()
    return values.iloc[-1] if len(values) else np.nan


def training_progress(run: dict) -> pd.DataFrame:
    frame = run["train"]
    columns = [
        "train/time",
        "train/batches",
        "train/samples",
        "train/device/samples_per_sec",
    ]
    missing = [
        column
        for column in ["step", *columns[:3]]
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют метрики ThroughputMonitor: "
            + ", ".join(missing)
        )

    available = [column for column in columns if column in frame.columns]
    progress = (
        frame[["step", *available]]
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in available})
        .dropna(subset=["train/time", "train/samples"])
        .sort_values("train/samples")
        .reset_index(drop=True)
    )
    return progress


def training_loss_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    progress_columns = [
        "train/time",
        "train/batches",
        "train/samples",
    ]

    if "train/data_loss" in frame.columns:
        loss_columns = ["train/data_loss"]
    elif "train/loss" in frame.columns:
        loss_columns = ["train/loss"]
    elif {"train/log_c", "train/log_v"}.issubset(frame.columns):
        loss_columns = ["train/log_c", "train/log_v"]
    else:
        raise KeyError(
            "В train CSV нет ни train/loss ни пары train/log_c и train/log_v"
        )

    required = ["step", *progress_columns, *loss_columns]
    missing = [column for column in required if column not in frame.columns]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют колонки: " + ", ".join(missing)
        )

    extra_columns = ["train/loss"] if "train/data_loss" in frame.columns and "train/loss" in frame.columns else []
    value_columns = list(dict.fromkeys([*progress_columns, *loss_columns, *extra_columns]))
    history = (
        frame[["step", *value_columns]]
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in value_columns})
        .dropna(subset=[*progress_columns, *loss_columns])
        .sort_values("train/samples")
        .reset_index(drop=True)
    )

    if len(loss_columns) == 1:
        history["loss"] = history[loss_columns[0]]
    else:
        history["loss"] = history["train/log_c"] - history["train/log_v"]

    if extra_columns:
        history["regularized_loss"] = history["train/loss"]

    return history.rename(
        columns={
            "train/time": "train_time_seconds",
            "train/batches": "batches_seen",
            "train/samples": "samples_seen",
        }
    )


In [ ]:
loss_histories = {
    label: training_loss_history(run)
    for label, run in runs.items()
}

for label, history in loss_histories.items():
    print(
        label,
        f"loss points={len(history)}",
        f"batches={int(history['batches_seen'].min()):,}..{int(history['batches_seen'].max()):,}",
        f"samples={int(history['samples_seen'].min()):,}..{int(history['samples_seen'].max()):,}",
    )


In [ ]:
fig, axis = plt.subplots(figsize=(10, 5))

for label, history in loss_histories.items():
    axis.plot(
        history["samples_seen"],
        history["loss"],
        marker=".",
        markersize=3,
        linewidth=1.3,
        color=MODEL_COLORS[label],
        label=f"{label} {len(history)} точек",
    )

axis.set_xscale("log")
axis.set_xlabel("Обработано объектов")
axis.set_ylabel("Train loss")
axis.set_title(f"Train loss по всем записям d={DIMENSION}")
axis.legend()
plt.tight_layout()
plt.show()


## Условные validation-метрики по объёму обработанных данных

Эти метрики существуют только в моменты validation


In [ ]:
VALIDATION_COLUMNS = [
    "val/cond_shape_score_forward",
    "val/cond_trend_score_forward",
]


def validation_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    missing = [
        column
        for column in ["step", *VALIDATION_COLUMNS]
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют validation-колонки: "
            + ", ".join(missing)
        )

    history = (
        frame[["step", *VALIDATION_COLUMNS]]
        .dropna(subset=VALIDATION_COLUMNS, how="all")
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in VALIDATION_COLUMNS})
        .sort_values("step")
        .reset_index(drop=True)
    )

    batch_size = int(run["config"]["data"]["batch_size"])
    accumulate = int(
        run["config"].get("trainer", {}).get("accumulate_grad_batches", 1)
    )
    history["expected_samples"] = (
        (history["step"].astype(int) + 1)
        * batch_size
        * accumulate
    ).astype(float)

    progress = training_progress(run).copy()
    progress["train/samples"] = progress["train/samples"].astype(float)
    history = pd.merge_asof(
        history.sort_values("expected_samples"),
        progress,
        left_on="expected_samples",
        right_on="train/samples",
        direction="nearest",
    )
    history["samples_seen"] = history["train/samples"].astype(int)
    history["train_time_seconds"] = history["train/time"]
    return history


In [ ]:
histories = {
    label: validation_history(run)
    for label, run in runs.items()
}

for label, history in histories.items():
    print(
        label,
        f"validation points={len(history)}",
        f"samples={history['samples_seen'].min():,}..{history['samples_seen'].max():,}",
    )


In [ ]:
validation_plots = [
    ("val/cond_shape_score_forward", "Conditional shape score"),
    ("val/cond_trend_score_forward", "Conditional trend score"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, (column, title) in zip(axes, validation_plots):
    for label, history in histories.items():
        points = history.dropna(subset=[column])
        axis.plot(
            points["samples_seen"],
            points[column],
            marker="o",
            color=MODEL_COLORS[label],
            label=label,
        )

    axis.set_xscale("log")
    axis.set_title(title)
    axis.set_xlabel("Обработано объектов")
    axis.legend()

axes[0].set_ylabel("Значение метрики")
fig.suptitle(f"Validation-метрики d={DIMENSION}", fontsize=14)
plt.tight_layout()
plt.show()


## Train loss по чистому времени обучения

Используются те же полные train-записи loss


In [ ]:
fig, axis = plt.subplots(figsize=(10, 5))

for label, history in loss_histories.items():
    axis.plot(
        history["train_time_seconds"],
        history["loss"],
        marker=".",
        markersize=3,
        linewidth=1.3,
        color=MODEL_COLORS[label],
        label=f"{label} {len(history)} точек",
    )

axis.set_xlabel("Чистое время обучения секунд")
axis.set_ylabel("Train loss")
axis.set_title(f"Train loss по времени d={DIMENSION}")
axis.legend()
plt.tight_layout()
plt.show()


## Условные validation-метрики по чистому времени обучения

`train/time` формируется callback `ThroughputMonitor` и не включает время validation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, (column, title) in zip(axes, validation_plots):
    for label, history in histories.items():
        points = history.dropna(subset=[column, "train_time_seconds"])
        axis.plot(
            points["train_time_seconds"],
            points[column],
            marker="o",
            color=MODEL_COLORS[label],
            label=label,
        )

    axis.set_title(title)
    axis.set_xlabel("Чистое время обучения секунд")
    axis.legend()

axes[0].set_ylabel("Значение метрики")
fig.suptitle(f"Validation-метрики по времени d={DIMENSION}", fontsize=14)
plt.tight_layout()
plt.show()


## Чистое время обучения по объёму обработанных данных


In [ ]:
progress = {
    label: training_progress(run)
    for label, run in runs.items()
}

fig, axis = plt.subplots(figsize=(9, 5))

for label, frame in progress.items():
    axis.plot(
        frame["train/samples"],
        frame["train/time"],
        marker="o",
        color=MODEL_COLORS[label],
        label=label,
    )

axis.set_xscale("log")
axis.set_xlabel("Обработано объектов")
axis.set_ylabel("Чистое время обучения секунд")
axis.set_title("Время обучения по объёму данных")
axis.legend()
plt.tight_layout()
plt.show()


## Скорость обучения

Среднее время итерации и throughput рассчитаны из накопленного train-времени без validation


In [ ]:
performance_rows = {}

for label, frame in progress.items():
    final = frame.iloc[-1]
    seconds = float(final["train/time"])
    batches = int(final["train/batches"])
    samples = int(final["train/samples"])

    performance_rows[label] = {
        "train_time_seconds": seconds,
        "iterations": batches,
        "samples": samples,
        "seconds_per_iteration": seconds / batches,
        "samples_per_second": samples / seconds,
        "samples_per_minute": 60 * samples / seconds,
    }

performance = pd.DataFrame.from_dict(performance_rows, orient="index")
display(performance.style.format(precision=3).set_caption("Скорость обучения"))


## Использование памяти GPU во время обучения

Показан накопленный пик памяти PyTorch CUDA allocator после train-update

Allocated peak соответствует максимальной памяти занятой тензорами с начала процесса

Reserved peak дополнительно включает максимальный размер кэша CUDA allocator


In [ ]:
GPU_MEMORY_COLUMN_PAIRS = [
    ("gpu/allocated_bytes_peak", "gpu/reserved_bytes_peak"),
    (
        "DeviceStatsMonitor.on_train_batch_end/allocated_bytes.all.peak",
        "DeviceStatsMonitor.on_train_batch_end/reserved_bytes.all.peak",
    ),
]


def gpu_memory_history(run: dict) -> pd.DataFrame | None:
    frame = run["train"]
    memory_columns = next(
        (pair for pair in GPU_MEMORY_COLUMN_PAIRS if set(pair).issubset(frame.columns)),
        None,
    )

    if memory_columns is None:
        return None

    allocated_column, reserved_column = memory_columns
    required = ["step", allocated_column, reserved_column]

    memory = (
        frame[required]
        .dropna(subset=[allocated_column, reserved_column], how="all")
        .groupby("step", as_index=False)
        .agg(
            {
                allocated_column: last_non_null,
                reserved_column: last_non_null,
            }
        )
        .sort_values("step")
    )
    train_progress = training_progress(run).sort_values("step")
    memory = pd.merge_asof(
        memory,
        train_progress[["step", "train/samples"]],
        on="step",
        direction="nearest",
    )
    memory["allocated_mib"] = memory[allocated_column] / 1024**2
    memory["reserved_mib"] = memory[reserved_column] / 1024**2
    return memory


In [ ]:
memory_histories = {}

for label, run in runs.items():
    history = gpu_memory_history(run)
    if history is None:
        print(f"{label}: GPU memory-метрики отсутствуют")
    else:
        memory_histories[label] = history

if memory_histories:
    model_count = len(memory_histories)
    fig, axes = plt.subplots(
        1,
        model_count,
        figsize=(7 * model_count, 5),
        squeeze=False,
    )
    axes = axes.ravel()

    for axis, (label, frame) in zip(axes, memory_histories.items()):
        axis.plot(
            frame["train/samples"],
            frame["allocated_mib"],
            label="Allocated peak",
            color=MODEL_COLORS[label],
        )
        axis.plot(
            frame["train/samples"],
            frame["reserved_mib"],
            label="Reserved peak",
            color=MODEL_COLORS[label],
            linestyle="--",
            alpha=0.8,
        )
        axis.set_xscale("log")
        axis.set_title(label)
        axis.set_xlabel("Обработано объектов")
        axis.set_ylabel("GPU memory MiB")
        axis.legend()

    fig.suptitle("Пиковая память GPU во время обучения", fontsize=14)
    plt.tight_layout()
    plt.show()


## Параметры последнего checkpoint

Для каждой модели загружается `checkpoints/last.ckpt` из выбранного архива

Веса компонент и содержимое CP-ядер показаны отдельно


In [ ]:
import torch


checkpoint_parameters = {}

for label, run in runs.items():
    checkpoint_path = run["run_dir"] / "checkpoints" / "last.ckpt"
    # Выбранные архивы содержат checkpoint наших локальных запусков
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state = checkpoint["state_dict"]
    log_alpha = state["log_alpha"].detach().to(torch.float64)
    if "log_cp_cores" in state:
        log_cores = state["log_cp_cores"].detach().to(torch.float64)
    else:
        keys = sorted(
            (key for key in state if key.startswith("log_cp_cores.")),
            key=lambda key: int(key.rsplit(".", 1)[1]),
        )
        log_cores = torch.stack([state[key].detach().to(torch.float64) for key in keys])
    if log_alpha.ndim != 1 or log_cores.ndim != 3 or log_cores.shape[-1] != len(log_alpha):
        raise ValueError(f"{label}: ожидались формы [K] и [D, S, K]")
    for value in (log_alpha, log_cores):
        if torch.isnan(value).any() or torch.isposinf(value).any():
            raise ValueError(f"{label}: параметры содержат NaN или +inf")
    checkpoint_parameters[label] = {
        "log_alpha": log_alpha.numpy(),
        "log_cores": log_cores.numpy(),
        "epoch": int(checkpoint["epoch"]) + 1,
        "step": int(checkpoint["global_step"]),
    }
    print(f"{label}: эпоха {checkpoint['epoch'] + 1}, шаг {checkpoint['global_step']}, "
          f"log_alpha {tuple(log_alpha.shape)}, log_cp_cores {tuple(log_cores.shape)}")
    del checkpoint, state


### Исходные ненормированные параметры

`log_alpha[k]` — натуральный логарифм ненормированного веса компоненты

`log_cp_cores[d, s, k]` — натуральный логарифм ненормированного элемента CP-ядра для координаты d категории s и компоненты k

Показываются непосредственно параметры из checkpoint без softmax без вычитания нормировочных констант и без экспоненцирования

Веса в обычной шкале равны экспонентам этих параметров — здесь они остаются в логарифмическом представлении чтобы сохранить экстремальные значения


### Эталон для сравнения нормированных весов

Читаем исходные `log_alpha` и `log_cp_cores` из сохранённого benchmark по конфигу каждого архива

Эталонные параметры используются только после канонической нормировки компонент

Это эталон распределения для задачи без штрафа а не гарантированный минимум регуляризованной эмпирической функции

CP-параметризация неоднозначна по перестановкам и масштабу компонент поэтому исходные коэффициенты напрямую не сравниваются

Число компонент эталона может отличаться от числа компонент обучаемой модели


In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import LocalEntryNotFoundError
from omegaconf import OmegaConf
from safetensors import safe_open


# Только чтение сохранённых параметров — без BenchmarkHD и вычисления prior
reference_sources = {}
reference_parameters = {}
for label, run in runs.items():
    config = OmegaConf.create(run["config"])
    spec = OmegaConf.to_container(config.data.benchmark, resolve=True)
    source = (
        str(spec["pretrained_model_name_or_path"]),
        str(spec.get("subfolder", "")),
        spec.get("revision"),
    )
    reference_sources.setdefault(source, []).append(label)

for index, ((repo, subfolder, revision), labels) in enumerate(reference_sources.items(), 1):
    local_file = Path(repo).expanduser() / subfolder / "model.safetensors"
    if Path(repo).expanduser().is_dir():
        if not local_file.is_file():
            raise FileNotFoundError(local_file)
        weights_path = local_file.resolve()
    else:
        kwargs = dict(repo_id=repo, subfolder=subfolder or None,
                      filename="model.safetensors", revision=revision)
        try:
            weights_path = Path(hf_hub_download(**kwargs, local_files_only=True))
        except LocalEntryNotFoundError:
            weights_path = Path(hf_hub_download(**kwargs))
    with safe_open(str(weights_path), framework="pt", device="cpu") as saved:
        reference_alpha = saved.get_tensor("log_alpha").to(torch.float64).numpy()
        reference_cores = saved.get_tensor("log_cp_cores").to(torch.float64).numpy()
    if (reference_alpha.ndim != 1 or reference_cores.ndim != 3
            or reference_cores.shape[-1] != len(reference_alpha)):
        raise ValueError("Некорректные формы эталонных параметров")
    if reference_cores.shape[0] != DIMENSION:
        raise ValueError("Размерность эталона отличается от архивов")
    for label in labels:
        if reference_cores.shape[:2] != checkpoint_parameters[label]["log_cores"].shape[:2]:
            raise ValueError(f"{label}: координаты или категории не совпадают с эталоном")
    for values in (reference_alpha, reference_cores):
        if np.isnan(values).any() or np.isposinf(values).any():
            raise ValueError("Эталон содержит NaN или +inf")
    reference_label = "Reference" if len(reference_sources) == 1 else f"Reference {index}"
    reference_parameters[reference_label] = {
        "log_alpha": reference_alpha,
        "log_cores": reference_cores,
        "description": "эталон распределения",
    }
    MODEL_COLORS[reference_label] = "#9467bd"
    print(f"{reference_label}: {repo}/{subfolder} · revision={revision or 'не закреплена'}")
    print(f"Файл: {weights_path}")
    print(f"Для: {', '.join(labels)} · log_alpha {reference_alpha.shape} · log_cp_cores {reference_cores.shape}")
    if revision is None and not Path(repo).expanduser().is_dir():
        print("Ревизия в архиве не закреплена — используется доступная версия benchmark из кеша или Hub")


In [ ]:
# Исходные параметры показываем только для обученных моделей
parameter_views = checkpoint_parameters


### Шкалы отображения

Ось значений и цветовая шкала используют symlog — линейную область около нуля и логарифмическую вне неё

Подписи делений показывают сами значения логарифмов весов а не их экспоненты

Общие пределы шкал для обеих моделей сохраняют сравнимость масштабов

Серые клетки и метки −∞ обозначают логарифм нулевого веса

Индексы компонент сохраняются и не означают соответствия компонент между моделями


In [ ]:
COMPONENTS_PER_PAGE = 8
SYMLOG_LINTHRESH = 1.0


### Ненормированные веса компонент — натуральные логарифмы


In [ ]:
fig, axes = plt.subplots(
    1, len(parameter_views), figsize=(7 * len(parameter_views), 4),
    squeeze=False, sharey=True,
)

for axis, (label, parameters) in zip(axes.ravel(), parameter_views.items()):
    values = parameters["log_alpha"]
    indices = np.arange(len(values))
    finite = np.isfinite(values)
    axis.bar(indices[finite], values[finite], color=MODEL_COLORS[label])
    for k in indices[~finite]:
        axis.text(k, 0.02, "−∞", transform=axis.get_xaxis_transform(), ha="center")
    axis.set_xticks(indices)
    axis.set_xlabel("Индекс компоненты")
    axis.set_ylabel("log_alpha = ln(ненормированный вес) · шкала symlog")
    axis.set_yscale("symlog", linthresh=SYMLOG_LINTHRESH)
    subtitle = parameters.get("description") or f"эпоха {parameters['epoch']} · шаг {parameters['step']}"
    axis.set_title(f"{label} · {subtitle}")
fig.tight_layout()
plt.show()


### Нормированные веса компонент и сравнение с эталоном

Для каждой компоненты каждое CP-ядро нормируется по категориям а снятая масса переносится в вес компоненты

После этого веса нормируются до суммы 1 поэтому распределение заданное CP-разложением не меняется

Компоненты отсортированы по убыванию веса поскольку их исходные индексы не идентифицируемы

Число компонент обученной модели и эталона может различаться

Это веса канонически нормированного CP-разложения а не зависящие от входа posterior-веса при sampling


In [ ]:
def normalized_component_parameters(parameters: dict) -> dict:
    log_alpha = torch.as_tensor(parameters["log_alpha"], dtype=torch.float64)
    log_cores = torch.as_tensor(parameters["log_cores"], dtype=torch.float64)
    log_factor_masses = torch.logsumexp(log_cores, dim=1)  # [D, K]
    log_component_masses = log_alpha + log_factor_masses.sum(dim=0)
    if not torch.isfinite(log_component_masses).any():
        raise ValueError("Все компоненты имеют нулевую массу")
    weights = torch.softmax(log_component_masses, dim=0)
    valid_factors = torch.isfinite(log_factor_masses)
    log_core_probabilities = log_cores - log_factor_masses[:, None, :]
    core_probabilities = log_core_probabilities.exp()
    core_probabilities = torch.where(
        valid_factors[:, None, :], core_probabilities, torch.nan,
    )
    order = torch.argsort(weights, descending=True)
    return {
        "weights": weights.numpy(),
        "core_probabilities": core_probabilities.numpy(),
        "order": order.numpy(),
    }


normalized_components = {
    label: normalized_component_parameters(parameters)
    for label, parameters in {**checkpoint_parameters, **reference_parameters}.items()
}


In [ ]:
fig, axis = plt.subplots(figsize=(10, 5))

for label, normalized in normalized_components.items():
    sorted_weights = normalized["weights"][normalized["order"]]
    ranks = np.arange(1, len(sorted_weights) + 1)
    axis.plot(
        ranks, sorted_weights, marker="o", linewidth=1.5,
        color=MODEL_COLORS[label], label=f"{label} · K={len(sorted_weights)}",
    )

axis.set_xlabel("Ранг компоненты по убыванию веса")
axis.set_ylabel("Нормированный вес компоненты")
axis.set_xticks(np.arange(1, max(len(v["weights"]) for v in normalized_components.values()) + 1))
axis.set_ylim(bottom=0)
axis.set_title("Распределение нормированных весов компонент")
axis.legend()
fig.tight_layout()
plt.show()


### Нормированные вероятности внутри CP-ядер

В каждой координате каждой компоненты вероятности категорий суммируются до 1

Масса снятая при нормировке ядра уже учтена в нормированном весе компоненты

Компоненты показаны в порядке убывания этих весов

Все модели и эталон используют одну цветовую шкалу

Серым отмечается ядро нулевой массы для которого условные вероятности не определены


In [ ]:
displayed_core_probabilities = [
    normalized["core_probabilities"][:, :, normalized["order"][:4]]
    for normalized in normalized_components.values()
]
finite_core_probabilities = np.concatenate([
    values[np.isfinite(values)] for values in displayed_core_probabilities
])
core_probability_vmax = float(finite_core_probabilities.max())
probability_cmap = plt.get_cmap("viridis").copy()
probability_cmap.set_bad("#bdbdbd")

for label, normalized in normalized_components.items():
    values = normalized["core_probabilities"]
    weights = normalized["weights"]
    order = normalized["order"][:4]
    dimension = values.shape[0]
    for first_rank in range(0, len(order), COMPONENTS_PER_PAGE):
        selected = order[first_rank:first_rank + COMPONENTS_PER_PAGE]
        ncols = min(4, len(selected))
        nrows = (len(selected) + ncols - 1) // ncols
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(5 * ncols, 3.8 * nrows),
            squeeze=False, layout="constrained",
        )
        used_axes = []
        for offset, (axis, component) in enumerate(zip(axes.ravel(), selected)):
            rank = first_rank + offset + 1
            picture = axis.imshow(
                np.ma.masked_invalid(values[:, :, component]),
                aspect="auto", interpolation="nearest", cmap=probability_cmap,
                vmin=0, vmax=core_probability_vmax,
            )
            axis.grid(False)
            axis.set_title(
                f"Ранг {rank} · исходная k={component} · вес={weights[component]:.4f}"
            )
            axis.set_xlabel("Категория")
            axis.set_ylabel("Координата")
            axis.set_yticks(np.arange(dimension))
            used_axes.append(axis)
        for axis in axes.ravel()[len(selected):]:
            axis.set_visible(False)
        fig.colorbar(
            picture, ax=used_axes, shrink=0.85,
            label="Нормированная вероятность категории внутри ядра",
        )
        last_rank = first_rank + len(selected)
        fig.suptitle(f"{label} · компоненты по весу · ранги {first_rank + 1}–{last_rank}")
        plt.show()


### Ненормированные элементы CP-ядер — натуральные логарифмы

Каждая тепловая карта показывает все координаты и категории одной компоненты

Общая цветовая шкала показывает абсолютный масштаб исходных логарифмов элементов ядер у обеих моделей


In [ ]:
from matplotlib.colors import SymLogNorm


finite_values = np.concatenate([
    p["log_cores"][np.isfinite(p["log_cores"])] for p in parameter_views.values()
])
extent = max(SYMLOG_LINTHRESH, float(np.max(np.abs(finite_values)))) if finite_values.size else SYMLOG_LINTHRESH
core_norm = SymLogNorm(linthresh=SYMLOG_LINTHRESH, vmin=-extent, vmax=extent)
core_cmap = plt.get_cmap("coolwarm").copy()
core_cmap.set_bad("#bdbdbd")

for label, parameters in parameter_views.items():
    values = parameters["log_cores"]
    dimension, categories, components = values.shape
    component_order = normalized_components[label]["order"]
    for first in range(0, components, COMPONENTS_PER_PAGE):
        selected = component_order[first:first + COMPONENTS_PER_PAGE]
        ncols = min(4, len(selected))
        nrows = (len(selected) + ncols - 1) // ncols
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(5 * ncols, 3.8 * nrows),
            squeeze=False, layout="constrained",
        )
        used_axes = []
        for axis, k in zip(axes.ravel(), selected):
            picture = axis.imshow(
                np.ma.masked_invalid(values[:, :, k]), aspect="auto",
                interpolation="nearest", cmap=core_cmap, norm=core_norm,
            )
            axis.grid(False)
            weight = normalized_components[label]["weights"][k]
            rank = first + len(used_axes) + 1
            axis.set_title(f"Ранг {rank} · исходная k={k} · вес={weight:.4f}")
            axis.set_xlabel("Категория")
            axis.set_ylabel("Координата")
            axis.set_yticks(np.arange(dimension))
            used_axes.append(axis)
        for axis in axes.ravel()[len(selected):]:
            axis.set_visible(False)
        fig.colorbar(
            picture, ax=used_axes, shrink=0.85,
            label="log_cp_cores = ln(ненормированный элемент ядра) · шкала symlog",
        )
        subtitle = parameters.get("description") or f"шаг {parameters['step']}"
        fig.suptitle(f"{label} · компоненты по весу · ранги {first + 1}–{first + len(selected)} · {subtitle}")
        plt.show()


## Regularized loss

Сумма основного Loss и штрафа для регуляризованного MM

Этот график не используется для сравнения Loss без штрафа между моделями

In [ ]:
regularized_histories = {
    label: history.dropna(subset=["regularized_loss"])
    for label, history in loss_histories.items()
    if "regularized_loss" in history.columns
}


In [ ]:
if regularized_histories:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label, history in regularized_histories.items():
        for axis, x in zip(axes, ["samples_seen", "train_time_seconds"]):
            axis.plot(history[x], history["regularized_loss"], color=MODEL_COLORS[label], label=label)
    axes[0].set_xscale("log")
    axes[0].set_xlabel("Samples seen")
    axes[1].set_xlabel("Train time (seconds)")
    for axis in axes:
        axis.set_ylabel("Regularized loss (loss + penalty)")
        axis.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No regularized loss in the loaded runs")
